# Historical Feature Engineering

Tujuan: Membangun **fitur historis** (*historical features*) yang merepresentasikan performa masing-masing tim sebelum pertandingan dimulai. Berbeda dengan *feature engineering* pada tahap sebelumnya yang hanya menggunakan informasi dasar seperti identitas tim, turnamen, dan status kandang, tahap ini mulai memanfaatkan riwayat pertandingan untuk menangkap kekuatan serta performa terkini setiap tim. Fitur-fitur yang dibangun diharapkan mampu memberikan informasi tambahan kepada model, misalnya mengenai konsistensi kemenangan, produktivitas mencetak gol, maupun kemampuan bertahan suatu tim. Seluruh fitur historis akan dihitung hanya menggunakan pertandingan yang telah terjadi sebelum pertandingan yang sedang diproses sehingga tetap merepresentasikan kondisi nyata (*real-world prediction*) dan terhindar dari *data leakage*. Output dari notebook ini adalah dataset baru yang berisi fitur baseline beserta fitur historis, sehingga siap digunakan pada proses pemodelan berikutnya.

## 1. Loading

Memuat dataset `results_clean` yang telah disimpan.

In [47]:
import pandas as pd
import numpy as np

In [48]:
matches = pd.read_csv(
    "../data/interim/results_clean.csv",
    parse_dates=["date"]
)

matches = matches[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament",
        "neutral"
    ]
].copy()

In [49]:
matches.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False


Dataset baseline berhasil dimuat dan akan digunakan sebagai dasar dalam membangun fitur historis. Seluruh pertandingan telah diurutkan secara kronologis sehingga informasi historis setiap tim dapat dihitung secara bertahap berdasarkan urutan waktu.

## 2. Konsep Historical Features

Pada tahap sebelumnya, model hanya memperoleh informasi dasar mengenai pertandingan, seperti identitas tim, jenis turnamen, dan status kandang. Meskipun informasi tersebut mampu memberikan kemampuan prediksi awal, model belum mengetahui bagaimana performa masing-masing tim sebelum pertandingan berlangsung.

Sebagai contoh, pertandingan berikut memiliki informasi dasar yang sama:

France vs Spain

Namun kenyataannya, kondisi kedua tim sebelum pertandingan dapat sangat berbeda.

Contoh:

France
5 pertandingan terakhir

W
W
D
W
W


Spain
5 pertandingan terakhir

L
D
W
L
D

Meskipun kedua pertandingan memiliki pasangan tim yang sama, peluang kemenangan tentu berbeda apabila salah satu tim sedang berada dalam performa yang jauh lebih baik.

Oleh karena itu, notebook ini membangun sejumlah fitur historis yang menggambarkan performa tim berdasarkan pertandingan-pertandingan sebelumnya. Seluruh perhitungan dilakukan secara kronologis sehingga setiap pertandingan hanya menggunakan informasi yang memang telah tersedia sebelum pertandingan tersebut dimainkan.

### Prinsip Utama

Dalam notebook ini, setiap fitur historis mengikuti prinsip berikut:
- Hanya menggunakan pertandingan yang terjadi sebelum pertandingan yang sedang diproses.
- Tidak menggunakan informasi hasil pertandingan yang sedang diprediksi.
- Seluruh perhitungan dilakukan secara kronologis berdasarkan tanggal pertandingan.
- Menghindari penggunaan informasi masa depan (*future information leakage*).
Dengan pendekatan ini, fitur yang dihasilkan dapat digunakan baik pada proses pelatihan model maupun pada prediksi pertandingan di dunia nyata.

## 3. Historical Feature Preparation
Kita harus melakukan persiapan terlebih dahulu.

### 3.1 Urutkan Data
Walaupun dataset hasil FE kemungkinan besar sudah urut, harus tetap divalidasi.

In [50]:
matches = (
    matches
    .sort_values("date")
    .reset_index(drop=True)
)

In [51]:
matches[["date", "home_team", "away_team"]].head()

,date,home_team,away_team
0,1872-11-30,Scotland,England
1,1873-03-08,England,Scotland
2,1874-03-07,Scotland,England
3,1875-03-06,England,Scotland
4,1876-03-04,Scotland,England


### 3.2 Membuat Struktur Penyimpanan Riwayat Tim
Kita perlu tempat menyimpan histori tiap negara.

In [52]:
from collections import defaultdict

In [53]:
team_history = defaultdict(list)

Dictionary `team_history` akan digunakan sebagai tempat menyimpan riwayat pertandingan setiap tim. Struktur ini memungkinkan proses perhitungan fitur historis dilakukan secara bertahap seiring iterasi kronologis pada dataset.

### 3.3 Menentukan Informasi yang Disimpan
Satu pertandingan akan menyimpan informasi apa saja?

| Kolom           | Alasan                   |
| --------------- | ------------------------ |
| Result          | Hitung winrate           |
| Goals Scored    | Avg goals                |
| Goals Conceded  | Avg conceded             |
| Goal Difference | Goal difference form     |
| Date            | Validasi kronologi/debug |

Setiap kali sebuah pertandingan selesai diproses, informasi penting dari pertandingan tersebut akan disimpan ke dalam riwayat masing-masing tim. Informasi ini nantinya digunakan untuk menghitung berbagai fitur historis seperti persentase kemenangan, rata-rata gol, maupun selisih gol pada pertandingan berikutnya.

### Insight : Historical Feature Preparation

Beberapa persiapan yang sudah dilakukan antara lain pengurutan data, membuat struktur penyimpanan riwayat statistik tim, dan menentukan informasi apa saja yang disimpan beserta alasan penyimpanan.

## 4. Historical Feature Construction
Kita akan mulai membuat pondasi historical features.

### 4.1 Menyiapkan Kolom Feature
Pertama, buat semua kolom.

In [54]:
historical_features = [
    "home_last5_winrate",
    "away_last5_winrate",
    "home_last10_winrate",
    "away_last10_winrate",
    "home_avg_goals_scored",
    "away_avg_goals_scored",
    "home_avg_goals_conceded",
    "away_avg_goals_conceded",
    "home_goal_difference_form",
    "away_goal_difference_form"
]

for col in historical_features:
    matches[col] = np.nan

Kolom-kolom fitur historis telah disiapkan pada dataset. Nilai setiap fitur akan diisi secara bertahap mengikuti urutan kronologis pertandingan sehingga tidak memanfaatkan informasi dari masa depan.

### 4.2 Helper Function
Mmebuat helper untuk efisiensi.

#### 4.2.1 Team History Structure

Setiap tim akan memiliki sebuah riwayat pertandingan (`team_history`) yang disimpan dalam bentuk *dictionary*. Nilai pada setiap tim berupa daftar (*list*) pertandingan sebelumnya yang berisi informasi penting seperti hasil pertandingan, jumlah gol yang dicetak, jumlah gol yang kebobolan, serta selisih gol.

Contoh struktur:

```python
team_history = {
    "France": [
        {
            "result": "W",
            "goals_scored": 2,
            "goals_conceded": 1,
            "goal_difference": 1
        },
        {
            "result": "D",
            "goals_scored": 1,
            "goals_conceded": 1,
            "goal_difference": 0
        }
    ],

    "Spain": [
        ...
    ]
}
```

Dengan struktur ini, seluruh fitur historis dapat dihitung menggunakan sumber data yang sama sehingga implementasi menjadi lebih konsisten dan mudah dikembangkan.

#### 4.2.2 Winrate Function

In [55]:
def calculate_winrate(history, last_n):
    """
    Calculate win rate from the last N matches.

    Parameters
    ----------
    history : list
        Match history of a team.
    last_n : int
        Number of recent matches to consider.

    Returns
    -------
    float
        Win rate between 0 and 1.
    """
    
    if len(history) == 0:
        return np.nan

    recent_matches = history[-last_n:]

    wins = sum(
        match["result"] == "W"
        for match in recent_matches
    )

    return wins / len(recent_matches)

Catatan: Fungsi tidak memaksa menggunakan 5 pertandingan untuk menghasilkan output. Hal ini bertujuan agar tim dengan pertandingan kurang dari 5 masih memiliki informasi.

#### 4.2.3 Average Goals Scored


In [56]:
def calculate_avg_scored(history, last_n):
    if len(history) == 0:
        return np.nan

    recent_matches = history[-last_n:]

    return np.mean([
        match["goals_scored"]
        for match in recent_matches
    ])

#### 4.2.4 Average Goals Conceded

In [57]:
def calculate_avg_conceded(history, last_n):
    if len(history) == 0:
        return np.nan

    recent_matches = history[-last_n:]

    return np.mean([
        match["goals_conceded"]
        for match in recent_matches
    ])

#### 4.2.5 Goal Difference

In [58]:
def calculate_goal_difference(history, last_n):
    if len(history) == 0:
        return np.nan

    recent_matches = history[-last_n:]

    return np.mean([
        match["goal_difference"]
        for match in recent_matches
    ])

Perhitungan setiap fitur historis dipisahkan ke dalam beberapa fungsi bantu (*helper functions*) agar kode lebih modular, mudah dipahami, dan dapat digunakan kembali apabila diperlukan pada tahap pengembangan berikutnya.

### 4.3 Building Historical Features
Mulai membangun historical features.

In [59]:
# %%
for idx, match in matches.iterrows():

    # -----------------------------
    # Match Information
    # -----------------------------
    home_team = match["home_team"]
    away_team = match["away_team"]

    home_score = match["home_score"]
    away_score = match["away_score"]

    # -----------------------------
    # Team History
    # -----------------------------
    home_history = team_history.get(home_team, [])
    away_history = team_history.get(away_team, [])

    # ==========================================================
    # HOME FEATURES
    # ==========================================================
    matches.loc[idx, "home_last5_winrate"] = calculate_winrate(
        home_history, 5
    )

    matches.loc[idx, "home_last10_winrate"] = calculate_winrate(
        home_history, 10
    )

    matches.loc[idx, "home_avg_goals_scored"] = calculate_avg_scored(
        home_history, 5
    )

    matches.loc[idx, "home_avg_goals_conceded"] = calculate_avg_conceded(
        home_history, 5
    )

    matches.loc[idx, "home_goal_difference_form"] = calculate_goal_difference(
        home_history, 5
    )

    # ==========================================================
    # AWAY FEATURES
    # ==========================================================
    matches.loc[idx, "away_last5_winrate"] = calculate_winrate(
        away_history, 5
    )

    matches.loc[idx, "away_last10_winrate"] = calculate_winrate(
        away_history, 10
    )

    matches.loc[idx, "away_avg_goals_scored"] = calculate_avg_scored(
        away_history, 5
    )

    matches.loc[idx, "away_avg_goals_conceded"] = calculate_avg_conceded(
        away_history, 5
    )

    matches.loc[idx, "away_goal_difference_form"] = calculate_goal_difference(
        away_history, 5
    )

    # ==========================================================
    # MATCH RESULT
    # ==========================================================
    if home_score > away_score:
        home_result = "W"
        away_result = "L"

    elif home_score < away_score:
        home_result = "L"
        away_result = "W"

    else:
        home_result = "D"
        away_result = "D"

    # ==========================================================
    # HOME RECORD
    # ==========================================================
    home_record = {
        "result": home_result,
        "goals_scored": home_score,
        "goals_conceded": away_score,
        "goal_difference": home_score - away_score
    }

    # ==========================================================
    # AWAY RECORD
    # ==========================================================
    away_record = {
        "result": away_result,
        "goals_scored": away_score,
        "goals_conceded": home_score,
        "goal_difference": away_score - home_score
    }

    # ==========================================================
    # UPDATE TEAM HISTORY
    # ==========================================================

    team_history[home_team].append(home_record)
    team_history[away_team].append(away_record)

### 4.4 Validasi dan Statistik

In [60]:
# %%
matches[
    [
        "date",
        "home_team",
        "away_team",
        "home_last5_winrate",
        "away_last5_winrate",
        "home_avg_goals_scored",
        "away_avg_goals_scored"
    ]
].head(20)

,date,home_team,away_team,home_last5_winrate,away_last5_winrate,home_avg_goals_scored,away_avg_goals_scored
0,1872-11-30,Scotland,England,NaN,NaN,NaN,NaN
1,1873-03-08,England,Scotland,0.000000,0.000000,0.000000,0.000000
2,1874-03-07,Scotland,England,0.000000,0.500000,1.000000,2.000000
3,1875-03-06,England,Scotland,0.333333,0.333333,1.666667,1.333333
4,1876-03-04,Scotland,England,0.250000,0.250000,1.500000,1.750000
5,1876-03-25,Scotland,Wales,0.400000,NaN,1.800000,NaN
6,1877-03-03,England,Scotland,0.200000,0.600000,1.400000,2.600000
7,1877-03-05,Wales,Scotland,0.000000,0.800000,0.000000,2.800000
8,1878-03-02,Scotland,England,0.800000,0.200000,2.800000,1.600000
9,1878-03-23,Scotland,Wales,1.000000,0.000000,3.800000,0.000000


In [62]:
matches[historical_features].describe()

,home_last5_winrate,away_last5_winrate,home_last10_winrate,away_last10_winrate,home_avg_goals_scored,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form
count,49292.000000,49238.000000,49292.000000,49238.000000,49292.000000,49238.000000,49292.000000,49238.000000,49292.000000,49238.000000
mean,0.390482,0.381769,0.390659,0.380805,1.485909,1.456864,1.454542,1.496211,0.031367,-0.039347
std,0.253621,0.252111,0.202937,0.202531,0.898580,0.886802,1.005434,1.056445,1.481583,1.517007
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-24.000000,-21.000000
25%,0.200000,0.200000,0.200000,0.200000,0.800000,0.800000,0.800000,0.800000,-0.800000,-0.800000
50%,0.400000,0.400000,0.400000,0.400000,1.400000,1.400000,1.200000,1.200000,0.000000,0.000000
75%,0.600000,0.600000,0.500000,0.500000,2.000000,1.800000,1.800000,1.800000,0.800000,0.800000
max,1.000000,1.000000,1.000000,1.000000,17.000000,21.000000,24.000000,21.000000,16.000000,21.000000


Selain itu, kita akan coba validasi manual contoh tim.

In [63]:
matches[
    (matches["home_team"] == "France") |
    (matches["away_team"] == "France")
].head(10)

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_last5_winrate,away_last5_winrate,home_last10_winrate,away_last10_winrate,home_avg_goals_scored,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form
163,1904-05-01,Belgium,France,3.0,3.0,Évence Coppée Trophy,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166,1905-02-12,France,Switzerland,1.0,0.0,Friendly,False,0.000000,NaN,0.000000,NaN,3.000000,NaN,3.000000,NaN,0.0,NaN
177,1905-05-07,Belgium,France,7.0,0.0,Friendly,False,0.000000,0.50,0.000000,0.500000,2.000000,2.00,3.500000,1.50,-1.5,0.50
190,1906-04-22,France,Belgium,0.0,5.0,Friendly,False,0.333333,0.25,0.333333,0.250000,1.333333,2.75,3.333333,2.75,-2.0,0.00
208,1907-04-21,Belgium,France,1.0,2.0,Friendly,False,0.600000,0.25,0.500000,0.250000,2.800000,1.00,1.800000,3.75,1.0,-2.75
215,1908-03-08,Switzerland,France,1.0,2.0,Friendly,False,0.000000,0.40,0.000000,0.400000,0.000000,1.20,1.000000,3.20,-1.0,-2.00
225,1908-04-12,France,Belgium,1.0,2.0,Friendly,False,0.600000,0.40,0.500000,0.500000,1.000000,1.60,2.800000,2.40,-1.8,-0.80
228,1908-05-10,Netherlands,France,4.0,1.0,Friendly,False,0.600000,0.40,0.625000,0.428571,2.600000,1.00,1.600000,3.20,1.0,-2.20
238,1908-10-22,France,Denmark,1.0,17.0,Olympic Games,True,0.400000,NaN,0.375000,NaN,1.200000,NaN,2.600000,NaN,-1.4,NaN
256,1909-05-09,Belgium,France,5.0,2.0,Friendly,False,0.400000,0.40,0.400000,0.333333,1.400000,1.40,2.600000,5.00,-1.2,-3.60


### 4.5 Missing Value Analysis
Sebagai validasi tambahan.

In [64]:
matches[historical_features].isna().sum()

home_last5_winrate           141
away_last5_winrate           195
home_last10_winrate          141
away_last10_winrate          195
home_avg_goals_scored        141
away_avg_goals_scored        195
home_avg_goals_conceded      141
away_avg_goals_conceded      195
home_goal_difference_form    141
away_goal_difference_form    195
dtype: int64

Missing value pada fitur historis bukan berasal dari data yang hilang, melainkan karena sebagian tim belum memiliki riwayat pertandingan yang cukup pada awal dataset. Hal ini merupakan kondisi yang wajar dan mencerminkan proses konstruksi fitur yang bebas dari future data leakage.

### 4.6 Save Dataset
Karena semua validasi valid, kita bisa menyimpan data hasilnya.

In [65]:
matches.to_csv(
    "../data/processed/historical_features.csv",
    index=False
)

### Insight : Historical Feature Construction

Seluruh fitur historis berhasil dibangun menggunakan satu proses iterasi kronologis. Pada setiap pertandingan, model terlebih dahulu mengambil riwayat kedua tim untuk menghitung seluruh fitur historis, kemudian baru memperbarui riwayat tersebut setelah pertandingan selesai diproses.

Pendekatan ini memastikan bahwa setiap fitur hanya memanfaatkan informasi yang tersedia sebelum pertandingan berlangsung sehingga terhindar dari *future data leakage*. Selain itu, penggunaan satu kali iterasi untuk membangun seluruh fitur membuat proses komputasi lebih efisien dan mudah dikembangkan ketika fitur historis baru ditambahkan pada tahap berikutnya.